# 001 · Defining Functions

> **Syntax › Functions** · Python 3.14

In C# every piece of code lives inside a class, so you write *methods*. Python has free-standing **functions** defined with `def`, and they are ordinary objects you can pass around like any other value.

**In this notebook you will learn:**

- how to define and call functions with `def`, with or without type hints
- what a function returns when there is no `return` (spoiler: `None`)
- how to return several values at once with a tuple
- that functions are first-class objects, and how that compares to C# delegates
- why Python has no method overloading, and what to use instead
- nested functions, docstrings, and the early-return style

## C# mental model

| C# | Python |
|---|---|
| `static int Add(int a, int b) { ... }` inside a class | `def add(a, b): ...` at module level |
| return type required (`void`, `int`, ...) | no return type required; hints are optional |
| `void` method | function that implicitly returns `None` |
| `(int, int)` tuple return / `out` params | return a tuple: `return a, b` |
| `Func<int, int>` / `Action` delegate | the function object itself |
| method overloading | not supported: last `def` wins |
| local functions (C# 7+) | nested functions |
| `/// <summary>` XML doc comments | docstrings (`"""..."""`) |

The key mental shift: `def` is an **executable statement** that creates a function object and binds it to a name, just like `x = 5` binds a name to an int.

## Defining and calling a function

```csharp
// C#
static string Greet(string name)
{
    return $"Hello, {name}!";
}

Console.WriteLine(Greet("alice"));
```

In Python, `def` is followed by the name, the parameter list, a colon, and an indented body. No braces, no access modifier, no `static`, no enclosing class.

In [ ]:
def greet(name):
    return f"Hello, {name}!"


print(greet("alice"))

## Type hints are optional

You can annotate parameters and the return type. The syntax is `name: type` for parameters and `-> type` for the return value. Like all type hints, they are **not enforced at runtime**: they are for your IDE and type checkers such as mypy or pyright *(see Type Hints › 001 · Type Annotations)*.

In [ ]:
def add(a: int, b: int) -> int:
    return a + b


print(add(2, 3))           # 5
print(add("ab", "cd"))     # "abcd": runs fine, only a type checker would complain
print(add.__annotations__) # the hints are just metadata stored on the function

## No `return` means `None`

A C# method declared `void` returns nothing. A Python function **always** returns something: if execution falls off the end (or hits a bare `return`), the result is `None` *(see Data Types › 005 · None)*.

```csharp
// C#
static void Log(string message) => Console.WriteLine($"[log] {message}");
// var x = Log("hi");   // compile error: cannot assign void
```

In [ ]:
def log(message: str) -> None:
    print(f"[log] {message}")


result = log("hi")      # legal in Python
print(result)           # None
print(result is None)   # True

⚠️ **Gotcha:** forgetting `return` is a silent bug. C# would refuse to compile `int Square(int x) { x * x; }`; Python happily returns `None`.

In [ ]:
def square(x):
    x * x               # computed... and thrown away


print(square(4))        # None, not 16

## Returning multiple values

C# gives you tuples or `out` parameters. In Python you simply `return a, b`: the comma builds a **tuple**, and the caller usually unpacks it straight into variables *(see Collections › 006 · Unpacking)*.

```csharp
// C#
static (int Min, int Max) MinMax(int[] values) => (values.Min(), values.Max());

var (lo, hi) = MinMax([3, 1, 4, 1, 5]);
```

In [ ]:
def min_max(values):
    return min(values), max(values)   # really: return (min(values), max(values))


lo, hi = min_max([3, 1, 4, 1, 5])
print(lo, hi)

pair = min_max([3, 1, 4])
print(pair, type(pair).__name__)       # it's a plain tuple

## Functions are first-class objects

In C# you need a delegate type (`Func<>`, `Action<>`, or a custom `delegate`) to store a method in a variable. In Python a function **is** an object: you can assign it, pass it, return it, and put it in collections without any wrapper type.

```csharp
// C#
Func<int, int> doubler = x => x * 2;
Func<int, int, int> op = Add;          // method group conversion
int Apply(Func<int, int> f, int v) => f(v);
```

In [ ]:
def double(x):
    return x * 2


f = double                      # no parentheses: we refer to the function, we don't call it
print(f(21))                    # 42
print(f is double)              # True: same object, two names
print(type(f).__name__, f.__name__)


def apply(func, value):
    return func(value)


print(apply(double, 5))         # pass a function as an argument

Storing functions in a dict is the idiomatic replacement for a `switch` that dispatches on a key. In C# you would write `Dictionary<string, Func<int, int, int>>`.

In [ ]:
def add(a, b): return a + b
def sub(a, b): return a - b
def mul(a, b): return a * b

operations = {"+": add, "-": sub, "*": mul}

for symbol in ["+", "-", "*"]:
    print(f"6 {symbol} 3 =", operations[symbol](6, 3))

⚠️ **Gotcha:** forgetting the parentheses does not call the function; you get the function object itself, which is truthy. `if is_ready:` instead of `if is_ready():` is always true.

In [ ]:
def is_ready():
    return False


print(is_ready)          # <function is_ready at 0x...>
print(bool(is_ready))    # True: a function object is truthy
print(is_ready())        # False: this is what you meant

## No method overloading

C# picks an overload at compile time based on parameter types. Python has no compile-time overload resolution: `def` just rebinds the name, so **the last definition wins**.

```csharp
// C#
static string Describe(int x)    => $"int {x}";
static string Describe(string s) => $"string {s}";
```

In [ ]:
def describe(x):
    return f"first version: {x}"


def describe(x, y):             # replaces the first one entirely
    return f"second version: {x}, {y}"


print(describe(1, 2))
try:
    describe(1)
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

What to use instead:

- **Default values** cover most "same name, fewer arguments" overloads *(see Functions › 002 · Parameters and Arguments)*.
- **`functools.singledispatch`** chooses an implementation based on the type of the **first** argument at runtime.
- `typing.overload` exists, but it only declares signatures for type checkers; you still write a single implementation.

In [ ]:
from functools import singledispatch


@singledispatch
def describe(x):
    return f"something else: {x!r}"


@describe.register
def _(x: int):
    return f"an int: {x}"


@describe.register
def _(x: str):
    return f"a string of length {len(x)}"


print(describe(42))
print(describe("hello"))
print(describe(3.5))

The `@` lines are *decorators*, covered in *Functions › 006 · Decorators*.

## Nested functions

You can define a function inside another function. It is the equivalent of a C# local function: visible only inside the enclosing function, and able to read its variables *(see Functions › 004 · Scope and Closures)*.

```csharp
// C#
static double Average(int[] values)
{
    int Total() => values.Sum();       // local function
    return (double)Total() / values.Length;
}
```

In [ ]:
def average(values):
    def total():                # only exists while average() runs
        return sum(values)      # can read the enclosing variable

    return total() / len(values)


print(average([2, 4, 6]))

try:
    total()
except NameError as e:
    print(f"{type(e).__name__}: {e}")

## Docstrings

A string literal as the **first statement** of a function becomes its docstring. Unlike C# XML comments, it is available at runtime through `__doc__` and `help()`. The common conventions (Google, NumPy, reST) are just formatting styles *(see Basics › 003 · Comments and Docstrings)*.

```csharp
// C#
/// <summary>Converts Celsius to Fahrenheit.</summary>
/// <param name="celsius">Temperature in °C.</param>
static double ToFahrenheit(double celsius) => celsius * 9 / 5 + 32;
```

In [ ]:
def to_fahrenheit(celsius: float) -> float:
    """Convert a temperature from Celsius to Fahrenheit.

    Args:
        celsius: Temperature in degrees Celsius.

    Returns:
        The temperature in degrees Fahrenheit.
    """
    return celsius * 9 / 5 + 32


print(to_fahrenheit(100))
print(to_fahrenheit.__doc__.splitlines()[0])   # first line of the docstring

## Early return and guard clauses

Early returns work exactly as in C#, and are just as idiomatic. They keep the "happy path" un-indented, which matters even more in Python where indentation *is* the block structure.

In [ ]:
def shipping_cost(weight_kg: float, express: bool = False) -> float | None:
    if weight_kg <= 0:
        return None              # guard clause
    if weight_kg > 30:
        return None              # too heavy to ship

    base = 5.0 + weight_kg * 1.2
    return base * 2 if express else base


print(shipping_cost(-1))
print(shipping_cost(10))
print(shipping_cost(10, express=True))

⚠️ **Gotcha:** a function that returns a value on some paths but falls off the end on others implicitly returns `None` there. C# would give you "not all code paths return a value"; Python gives you nothing. Type checkers will catch it if you annotate the return type.

In [ ]:
def sign(n: int) -> str:
    if n > 0:
        return "positive"
    if n < 0:
        return "negative"
    # forgot the zero case


print(sign(0))   # None

## Placeholder bodies

A function body cannot be empty. Use `pass` (do nothing) or `...` (the Ellipsis literal, common for stubs) while sketching code. To mark something as not implemented yet, raise `NotImplementedError` (C#: `throw new NotImplementedException()`).

In [ ]:
def todo():
    pass


def stub(): ...


def later():
    raise NotImplementedError("coming soon")


print(todo(), stub())
try:
    later()
except NotImplementedError as e:
    print(f"{type(e).__name__}: {e}")

## Summary

| Concept | C# | Python |
|---|---|---|
| Define | `static int Add(int a, int b) { ... }` | `def add(a, b): ...` |
| Return type | required | optional hint: `-> int` |
| No return value | `void` | implicitly returns `None` |
| Multiple results | `(int, int)` / `out` | `return a, b` (tuple) |
| Function in a variable | `Func<int, int> f = Double;` | `f = double` |
| Dispatch table | `Dictionary<string, Func<...>>` | `{"+": add, ...}` |
| Overloading | by signature | not supported; defaults or `@singledispatch` |
| Local function | local function | nested `def` |
| Documentation | `/// <summary>` | docstring, read via `__doc__` / `help()` |
| Not implemented | `throw new NotImplementedException()` | `raise NotImplementedError` |

## Exercises

1. Write `stats(values)` that returns the count, sum, and mean of a list as a tuple, and unpack the result into three variables.
2. Build a dispatch dict mapping `"upper"`, `"lower"` and `"title"` to functions, then apply each to `"hello world"`. *Hint:* `str.upper` is already a function.
3. Explain what this prints and why: `def f(): print("hi")` followed by `print(f())`.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
# 1 — return a tuple, unpack at the call site
def stats(values):
    total = sum(values)
    return len(values), total, total / len(values)

count, total, mean = stats([2, 4, 6, 8])

# 2 — unbound methods like str.upper are plain functions taking the string
transforms = {"upper": str.upper, "lower": str.lower, "title": str.title}
for name, fn in transforms.items():
    print(name, fn("hello world"))

# 3 — it prints "hi" (from inside f) and then "None",
#     because f has no return statement and so returns None.
```

</details>